<a href="https://colab.research.google.com/github/GaganDeshbhandari/ML-major-project-notebook/blob/main/factllama_qlora_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 2
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("No GPU detected. Select a GPU runtime.")


PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
# Cell 3
# %pip install -q -U transformers datasets peft trl accelerate bitsandbytes sentencepiece huggingface_hub
%pip install -q -U trl bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.1 MB/s eta 0:00:00


In [ ]:
# Cell 4
import torch
import transformers
import datasets
import peft
import trl
import accelerate
import bitsandbytes
import huggingface_hub

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

print("\nInstalled packages:")
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4

Installed packages:
Transformers: 5.18.0
Datasets: 4.8.5
PEFT: 0.21.1
TRL: 1.15.0
Accelerate: 1.15.0
BitsAndBytes: 0.50.2


In [ ]:
# Cell 5
# Run this only one time to link the system to the Hugging-Face
from huggingface_hub import login
login()

In [ ]:
# Cell 6
from huggingface_hub import model_info

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

info = model_info(MODEL_ID)

print("Model:", info.id)
print("Model access verified.")

Model: meta-llama/Llama-3.2-3B-Instruct
Model access verified.


In [ ]:
# Cell 7
from pathlib import Path

DATASET_ROOT = Path(
    "/content/drive/MyDrive/FactLLaMA Datasets"
)

RAWFC_DIR = DATASET_ROOT / "RAWFC"
LIAR_RAW_DIR = DATASET_ROOT / "LIAR-RAW"
DATA_DIR = DATASET_ROOT / "factllama_data"

print("RAWFC extracted:", RAWFC_DIR.exists())
print("LIAR-RAW extracted:", LIAR_RAW_DIR.exists())
print("Processed data:", DATA_DIR.exists())

for split in ["train", "val", "test"]:
    path = DATA_DIR / f"{split}.jsonl"
    print(f"{split}: exists={path.exists()}")

RAWFC extracted: True
LIAR-RAW extracted: True
Processed data: True
train: exists=True
val: exists=True
test: exists=True


In [ ]:
# Cell 8
import json
from pathlib import Path
from transformers import AutoTokenizer

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
DATA_DIR = DATASET_ROOT / "factllama_data"

# Load the tokenizer; this also checks model-repository access.
tok_loader = AutoTokenizer
tokenizer = tok_loader.from_pretrained(MODEL_ID)

SYSTEM_PROMPT = (
    "You are a fact-checking assistant. Evaluate the claim using the "
    "provided evidence. Return exactly one label: true, half, or false. "
    "Do not use outside knowledge."
)

def build_prompt(record, max_evidence=None):
    evidence = record["evidence"]
    if max_evidence is not None:
        evidence = evidence[:max_evidence]

    evidence_text = "\n".join(
        f"[{i + 1}] {item['text']}"
        for i, item in enumerate(evidence)
    )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Claim: {record['claim']}\n\n"
                f"Evidence:\n{evidence_text}\n\n"
                "Verdict:"
            )
        }
    ]

def count_tokens(messages):
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return len(tokenizer.encode(rendered, add_special_tokens=False))

with open(DATA_DIR / "train.jsonl", encoding="utf-8") as f:
    train_records = [json.loads(line) for line in f]

all_evidence_lengths = []
capped_lengths = []

for record in train_records:
    all_evidence_lengths.append(
        count_tokens(build_prompt(record))
    )
    capped_lengths.append(
        count_tokens(build_prompt(record, max_evidence=12))
    )

def summarize(name, values):
    values = sorted(values)
    n = len(values)
    print(f"\n{name}")
    print("Minimum:", values[0])
    print("Median:", values[n // 2])
    print("90th percentile:", values[int(n * 0.90)])
    print("Maximum:", values[-1])
    print("Prompts over 2048 tokens:", sum(x > 2048 for x in values))
    print("Prompts over 4096 tokens:", sum(x > 4096 for x in values))

print("Tokenizer loaded successfully:", MODEL_ID)
summarize("ALL EVIDENCE", all_evidence_lengths)
summarize("FIRST 12 EVIDENCE SENTENCES", capped_lengths)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer loaded successfully: meta-llama/Llama-3.2-3B-Instruct

ALL EVIDENCE
Minimum: 189
Median: 1317
90th percentile: 3135
Maximum: 5296
Prompts over 2048 tokens: 455
Prompts over 4096 tokens: 48

FIRST 12 EVIDENCE SENTENCES
Minimum: 189
Median: 595
90th percentile: 750
Maximum: 3082
Prompts over 2048 tokens: 1
Prompts over 4096 tokens: 0


In [ ]:
# Cell 9
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

assert torch.cuda.is_available(), "CUDA GPU is not available. Check Colab runtime type."

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GB")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\nModel loaded and LoRA adapters configured.")
print("Allocated GPU memory:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

GPU: Tesla T4
GPU memory: 14.56 GB


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511

Model loaded and LoRA adapters configured.
Allocated GPU memory: 2.91 GB


In [ ]:
# Cell 10
import json
from datasets import Dataset

DATA_DIR = DATASET_ROOT / "factllama_data"
MAX_LENGTH = 2048
MAX_EVIDENCE = 12
VALID_LABELS = {"true", "half", "false"}

SYSTEM_PROMPT = (
    "You are a fact-checking assistant. Evaluate the claim using only "
    "the provided evidence. Output exactly one verdict: true, half, or false."
)

def encode_record(record):
    label = record["label"].strip().lower()
    if label not in VALID_LABELS:
        raise ValueError(f"Unexpected label: {label}")

    evidence = record["evidence"][:MAX_EVIDENCE]
    evidence_text = "\n".join(
        f"[{i + 1}] {item['text']}" for i, item in enumerate(evidence)
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Claim: {record['claim']}\n\n"
                f"Evidence:\n{evidence_text}\n\n"
                "Verdict:"
            ),
        },
    ]

    # BatchEncoding exposes the token IDs through .input_ids.
    prompt_result = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
    )

    prompt_ids = prompt_result.input_ids

    # Convert tensors/arrays to a regular Python list if necessary.
    if hasattr(prompt_ids, "tolist"):
        prompt_ids = prompt_ids.tolist()

    if prompt_ids and isinstance(prompt_ids[0], list):
        prompt_ids = prompt_ids[0]

    prompt_ids = [int(x) for x in prompt_ids]

    target_ids = list(tokenizer.encode(
        label, add_special_tokens=False
    )) + [tokenizer.eos_token_id]

    prompt_ids = prompt_ids[:MAX_LENGTH - len(target_ids)]
    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
        "length": len(input_ids),
        "verdict": label,
    }

encoded_datasets = {}

for split in ["train", "val", "test"]:
    path = DATA_DIR / f"{split}.jsonl"

    with path.open(encoding="utf-8") as f:
        records = [json.loads(line) for line in f]

    encoded = [encode_record(record) for record in records]
    dataset = Dataset.from_list(encoded)
    encoded_datasets[split] = dataset

    print(f"\n{split.upper()}")
    print("Examples:", len(dataset))
    print("Maximum encoded length:", max(x["length"] for x in encoded))

assert len(encoded_datasets["train"]) == 1612
assert len(encoded_datasets["val"]) == 200
assert len(encoded_datasets["test"]) == 200

sample = encoded_datasets["train"][0]
supervised = [x for x in sample["labels"] if x != -100]

assert len(supervised) > 0
assert len(sample["input_ids"]) <= MAX_LENGTH
assert tokenizer.decode(
    supervised, skip_special_tokens=True
).strip() in VALID_LABELS

print("\nPreprocessing checks passed.")
print("Decoded sample target:",
      tokenizer.decode(supervised, skip_special_tokens=True))



TRAIN
Examples: 1612
Maximum encoded length: 2048

VAL
Examples: 200
Maximum encoded length: 1031


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



TEST
Examples: 200
Maximum encoded length: 1211

Preprocessing checks passed.
Decoded sample target: half


In [ ]:
# Cell 11


import torch

PAD_TOKEN_ID = (
    tokenizer.pad_token_id
    if tokenizer.pad_token_id is not None
    else tokenizer.eos_token_id
)

def collate_batch(features):
    max_len = max(len(item["input_ids"]) for item in features)

    input_ids = []
    attention_masks = []
    labels = []

    for item in features:
        padding = max_len - len(item["input_ids"])

        input_ids.append(
            item["input_ids"] + [PAD_TOKEN_ID] * padding
        )
        attention_masks.append(
            item["attention_mask"] + [0] * padding
        )
        labels.append(
            item["labels"] + [-100] * padding
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

# Verify that a batch is padded correctly
test_batch = collate_batch([
    encoded_datasets["train"][0],
    encoded_datasets["train"][1],
])

print("Input shape:", test_batch["input_ids"].shape)
print("Attention mask shape:", test_batch["attention_mask"].shape)
print("Labels shape:", test_batch["labels"].shape)
print("Ignored label count:", (test_batch["labels"] == -100).sum().item())

assert test_batch["input_ids"].shape == test_batch["labels"].shape
assert test_batch["attention_mask"].shape == test_batch["input_ids"].shape

print("Data collator checks passed.")


Input shape: torch.Size([2, 538])
Attention mask shape: torch.Size([2, 538])
Labels shape: torch.Size([2, 538])
Ignored label count: 1072
Data collator checks passed.


In [ ]:
# Cell 12


from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/FactLLaMA Datasets")
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
OUTPUT_DIR = PROJECT_DIR / "factllama_qlora_output"
FINAL_ADAPTER_DIR = PROJECT_DIR / "factllama_final_adapter"
LOG_DIR = PROJECT_DIR / "training_logs"

for directory in [
    CHECKPOINT_DIR,
    OUTPUT_DIR,
    FINAL_ADAPTER_DIR,
    LOG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Checkpoint directory:", CHECKPOINT_DIR)
print("Training output directory:", OUTPUT_DIR)
print("Final adapter directory:", FINAL_ADAPTER_DIR)
print("Directories created successfully.")


Checkpoint directory: /content/drive/MyDrive/FactLLaMA Datasets/checkpoints
Training output directory: /content/drive/MyDrive/FactLLaMA Datasets/factllama_qlora_output
Final adapter directory: /content/drive/MyDrive/FactLLaMA Datasets/factllama_final_adapter
Directories created successfully.


In [ ]:
# Cell 13


test_file = CHECKPOINT_DIR / "drive_write_test.txt"

test_file.write_text("FactLLaMA checkpoint storage is working.")
assert test_file.exists()
assert test_file.read_text() == "FactLLaMA checkpoint storage is working."

test_file.unlink()

print("Google Drive read/write test passed.")


Google Drive read/write test passed.


In [ ]:
# Cell 14


from transformers import TrainingArguments, Trainer

model.config.use_cache = False

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=19,  # Approximately 3% of one epoch's optimizer steps
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",

    # T4 GPU
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Validation
    eval_strategy="steps",
    eval_steps=50,

    # Checkpoints on Google Drive
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    save_only_model=False,

    # Select the best validation checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_strategy="steps",
    logging_steps=10,
    report_to="none",

    # Our custom collator handles padding
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_datasets["train"],
    eval_dataset=encoded_datasets["val"],
    data_collator=collate_batch,
    processing_class=tokenizer,
)

print("Trainer configured successfully.")
print("Training examples:", len(encoded_datasets["train"]))
print("Validation examples:", len(encoded_datasets["val"]))
print("Checkpoint directory:", training_args.output_dir)
print("Save interval:", training_args.save_steps, "steps")


Trainer configured successfully.
Training examples: 1612
Validation examples: 200
Checkpoint directory: /content/drive/MyDrive/FactLLaMA Datasets/factllama_qlora_output
Save interval: 50 steps


In [22]:
# Cell 15

import torch
from pathlib import Path

assert torch.cuda.is_available(), "CUDA GPU is unavailable."
assert len(encoded_datasets["train"]) == 1612
assert len(encoded_datasets["val"]) == 200
assert len(encoded_datasets["test"]) == 200

checkpoint_dir = Path(training_args.output_dir)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0))
print("Model:", model.config._name_or_path)
print("Train examples:", len(encoded_datasets["train"]))
print("Validation examples:", len(encoded_datasets["val"]))
print("Test examples:", len(encoded_datasets["test"]))
print("Checkpoint directory:", checkpoint_dir)
print("Checkpoint directory exists:", checkpoint_dir.exists())
print("Trainer initialized:", trainer is not None)
print("Setup verification passed.")


GPU: Tesla T4
Model: meta-llama/Llama-3.2-3B-Instruct
Train examples: 1612
Validation examples: 200
Test examples: 200
Checkpoint directory: /content/drive/MyDrive/FactLLaMA Datasets/factllama_qlora_output
Checkpoint directory exists: True
Trainer initialized: True
Setup verification passed.


In [ ]:
# Cell 16

from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))

if last_checkpoint:
    print("Resuming from checkpoint:", last_checkpoint)
    train_result = trainer.train(
        resume_from_checkpoint=last_checkpoint
    )
else:
    print("No existing checkpoint found. Starting fresh training.")
    train_result = trainer.train()

print("Training run completed.")


Resuming from checkpoint: /content/drive/MyDrive/FactLLaMA Datasets/factllama_qlora_output/checkpoint-200


Step,Training Loss,Validation Loss
250,0.371171,0.480126
